# OSRM Road Distance Calculator
Calculates road distances for 14,200 orders using the free OSRM public API.

**Steps:**
1. Upload your CSV when prompted
2. Wait ~5-10 minutes for processing
3. Download the result CSV with road distances

In [ ]:
# Step 1: Upload your CSV file
from google.colab import files
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f"\nUploaded: {INPUT_FILE}")

In [ ]:
# Step 2: Install dependencies & load data
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} orders from {df['Dark store'].nunique()} stores")
df.head()

In [ ]:
# Step 3: Define the distance calculation functions

OSRM_BASE = "https://router.project-osrm.org"
BATCH_SIZE = 50
DELAY = 1.5  # seconds between requests
RETRY_COUNT = 3
TIMEOUT = 30


def get_batch_distances(store_lat, store_lon, customer_coords):
    """Use OSRM Table API: one store -> many customers in one request."""
    coords = f"{store_lon},{store_lat}"
    for clat, clon in customer_coords:
        coords += f";{clon},{clat}"

    destinations = ";".join(str(i) for i in range(1, len(customer_coords) + 1))
    url = f"{OSRM_BASE}/table/v1/driving/{coords}?sources=0&destinations={destinations}&annotations=distance"

    for attempt in range(RETRY_COUNT):
        try:
            resp = requests.get(url, timeout=TIMEOUT)
            if resp.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue

            data = resp.json()
            if data.get("code") == "Ok":
                return [round(d / 1000, 2) if d is not None else None for d in data["distances"][0]]
            else:
                return fallback_individual(store_lat, store_lon, customer_coords)
        except Exception as e:
            if attempt < RETRY_COUNT - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return [None] * len(customer_coords)

    return [None] * len(customer_coords)


def fallback_individual(store_lat, store_lon, customer_coords):
    """Fallback: individual route queries if table API fails."""
    results = []
    for clat, clon in customer_coords:
        url = f"{OSRM_BASE}/route/v1/driving/{store_lon},{store_lat};{clon},{clat}?overview=false"
        try:
            time.sleep(1)
            resp = requests.get(url, timeout=TIMEOUT)
            data = resp.json()
            if data.get("code") == "Ok" and data.get("routes"):
                results.append(round(data["routes"][0]["distance"] / 1000, 2))
            else:
                results.append(None)
        except Exception:
            results.append(None)
    return results


# Test connection
test = requests.get(f"{OSRM_BASE}/route/v1/driving/77.5946,12.9716;77.6200,13.0000?overview=false", timeout=10)
if test.json().get("code") == "Ok":
    print("OSRM server is reachable! Ready to calculate distances.")
else:
    print("WARNING: OSRM server issue. Results may be incomplete.")

In [ ]:
# Step 4: Calculate all road distances (takes ~5-10 minutes)

store_groups = df.groupby("Dark store")
distances = {}  # order_id -> distance

total_orders = len(df)
pbar = tqdm(total=total_orders, desc="Calculating distances")

for store_name, group in store_groups:
    store_lat = group["Lat Store"].iloc[0]
    store_lon = group["Long Store"].iloc[0]

    order_ids = group["Order ID"].astype(str).tolist()
    cust_lats = group["Lat Order"].tolist()
    cust_lons = group["Long Order"].tolist()

    # Process in batches
    for i in range(0, len(order_ids), BATCH_SIZE):
        batch_ids = order_ids[i:i + BATCH_SIZE]
        batch_coords = list(zip(cust_lats[i:i + BATCH_SIZE], cust_lons[i:i + BATCH_SIZE]))

        batch_distances = get_batch_distances(store_lat, store_lon, batch_coords)

        for oid, dist in zip(batch_ids, batch_distances):
            distances[str(oid)] = dist

        pbar.update(len(batch_ids))
        time.sleep(DELAY)

pbar.close()
print(f"\nDone! Calculated {len(distances):,} distances.")

In [ ]:
# Step 5: Save results and download

df["road_distance_km"] = df["Order ID"].astype(str).map(lambda oid: distances.get(str(oid)))

success = df["road_distance_km"].notna().sum()
failed = df["road_distance_km"].isna().sum()
valid = df["road_distance_km"].dropna()

print(f"Results:")
print(f"  Successful: {success:,} ({success/len(df)*100:.1f}%)")
print(f"  Failed:     {failed:,} ({failed/len(df)*100:.1f}%)")
if len(valid) > 0:
    print(f"  Avg distance:  {valid.mean():.1f} km")
    print(f"  Max distance:  {valid.max():.1f} km")
    print(f"  Min distance:  {valid.min():.1f} km")
    print(f"  Median:        {valid.median():.1f} km")

OUTPUT_FILE = "orders_with_road_distance.csv"
df.to_csv(OUTPUT_FILE, index=False)

# Auto-download
files.download(OUTPUT_FILE)
print(f"\nDownloading {OUTPUT_FILE}...")

In [ ]:
# Optional: Quick visualization of distance distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(valid, bins=50, edgecolor='black', alpha=0.7, color='#2196F3')
axes[0].set_xlabel('Road Distance (km)')
axes[0].set_ylabel('Number of Orders')
axes[0].set_title('Distribution of Road Distances')
axes[0].axvline(valid.median(), color='red', linestyle='--', label=f'Median: {valid.median():.1f} km')
axes[0].legend()

# Top 10 stores by avg distance
store_avg = df.groupby('Dark store')['road_distance_km'].mean().sort_values(ascending=False).head(10)
axes[1].barh(range(len(store_avg)), store_avg.values, color='#FF9800')
axes[1].set_yticks(range(len(store_avg)))
axes[1].set_yticklabels([s.replace('_wh_hl_01', '') for s in store_avg.index], fontsize=8)
axes[1].set_xlabel('Avg Road Distance (km)')
axes[1].set_title('Top 10 Stores by Avg Delivery Distance')

plt.tight_layout()
plt.show()